In [0]:
 %run "/Workspace/Users/jeancarlosramoshuamanlazo@hotmail.com/databricksjc/Proceso/0_CONFIGURACION"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}")

In [0]:
autos_conformed = spark.table(f"{silver_catalog}.autos_conformed")

DIMENSIONES

In [0]:
dim_make = (
    autos_conformed
    .select("make")
    .filter(F.col("make").isNotNull())
    .dropDuplicates()
    .withColumn("make_id", F.row_number().over(Window.orderBy("make")))
)

save_delta_table(dim_make, gold_tables["dim_make"])

In [0]:
dim_body_style = (
    autos_conformed
    .select("body_style")
    .filter(F.col("body_style").isNotNull())
    .dropDuplicates()
    .withColumn("body_style_id", F.row_number().over(Window.orderBy("body_style")))
)

save_delta_table(dim_body_style, gold_tables["dim_body_style"])

In [0]:
dim_engine = (
    autos_conformed
    .select(
        "engine_type",
        "num_of_cylinders",
        "engine_size"
    )
    .dropDuplicates()
    .withColumn(
        "engine_id",
        F.row_number().over(
            Window.orderBy("engine_type", "num_of_cylinders", "engine_size")
        )
    )
)

save_delta_table(dim_engine, gold_tables["dim_engine"])

In [0]:
dim_fuel = (
    autos_conformed
    .select("fuel_system")
    .dropDuplicates()
    .withColumn(
        "fuel_id",
        F.row_number().over(Window.orderBy("fuel_system"))
    )
)

save_delta_table(dim_fuel, gold_tables["dim_fuel"])

In [0]:
dim_drive = (
    autos_conformed
    .select("drive_wheels", "aspiration")
    .dropDuplicates()
    .withColumn(
        "drive_id",
        F.row_number().over(Window.orderBy("drive_wheels", "aspiration"))
    )
)

save_delta_table(dim_drive, gold_tables["dim_drive"])

TABLA DE HECHOS - FACT

In [0]:
fact_joined = (
    autos_conformed.alias("a")
    .join(dim_make.alias("m"), "make", "left")
    .join(dim_body_style.alias("b"), "body_style", "left")
    .join(dim_drive.alias("d"), ["drive_wheels", "aspiration"], "left")
    .join(
        dim_engine.alias("e"),
        ["engine_type", "num_of_cylinders", "engine_size"],
        "left"
    )
    .join(
        dim_fuel.alias("f"),
        "fuel_system",
        "left"
    )
)

FINAL

In [0]:
fact_autos = (
    fact_joined
    .withColumn(
        "id_auto",
        F.row_number().over(Window.orderBy("make"))
    )
    .select(
        "id_auto",
        "make_id",
        "body_style_id",
        "engine_id",
        "fuel_id",
        "drive_id",

        "price",
        "horsepower",
        "city_mpg",
        "highway_mpg",
        "curb_weight"
    )
)

save_delta_table(fact_autos, gold_tables["fact_autos"])